In [1]:
# ============================================================
# CELL 0 – LOCAL ENVIRONMENT SETUP
# ============================================================
import os, sys

# Cấu trúc Local-first: lấy thư mục gốc của project
# Vì notebook nằm trong thư mục notebooks/, nên PROJECT_ROOT là thư mục cha.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
REPO_PATH = PROJECT_ROOT

# Add src/ to Python path
if os.path.join(PROJECT_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

print('✅ Local Environment ready. PROJECT_ROOT:', PROJECT_ROOT)


✅ Local Environment ready. PROJECT_ROOT: d:\bt_lap_trinh\NLP\Ứng dụng\Project\hate-speech-detection


In [2]:
# ============================================================
# CELL 1 - TRAIN BASELINE MODEL
# ============================================================
import os
import json
import joblib
import pandas as pd
from omegaconf import OmegaConf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from data.preprocessing import resolve_project_path

cfg = OmegaConf.load(os.path.join(REPO_PATH, "configs/paths.yaml"))
project_root = PROJECT_ROOT
label_names = [cfg.model.label_map[i] for i in range(int(cfg.model.num_labels))]

train_path = resolve_project_path(cfg.data.train_processed, project_root)
val_path = resolve_project_path(cfg.data.val_processed, project_root)
model_path = resolve_project_path(cfg.models.baseline, project_root)
report_path = resolve_project_path(cfg.results.baseline_report, project_root)

train = pd.read_parquet(train_path)
dev = pd.read_parquet(val_path)
required = {"text", "label"}
for name, df in [("train", train), ("dev", dev)]:
    missing = required - set(df.columns)
    assert not missing, f"{name} is missing columns: {missing}; found {list(df.columns)}"

print(f"Data loaded: train={len(train):,}, dev={len(dev):,}")

tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    max_features=50000,
    sublinear_tf=True,
)
X_train = tfidf.fit_transform(train["text"])
X_dev = tfidf.transform(dev["text"])

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, train["label"])

y_pred = model.predict(X_dev)
macro_f1 = f1_score(dev["label"], y_pred, average="macro", zero_division=0)
report = classification_report(
    dev["label"],
    y_pred,
    labels=list(range(len(label_names))),
    target_names=label_names,
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(dev["label"], y_pred, labels=list(range(len(label_names)))).tolist()

print(f"Baseline Macro F1 Score: {macro_f1:.4f}")
print(classification_report(
    dev["label"],
    y_pred,
    labels=list(range(len(label_names))),
    target_names=label_names,
    zero_division=0,
))

model_path.parent.mkdir(parents=True, exist_ok=True)
report_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"vectorizer": tfidf, "model": model}, model_path)

with report_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "evaluated_split": "dev",
            "macro_f1": float(macro_f1),
            "classification_report": report,
            "confusion_matrix": cm,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print(f"Baseline artifacts saved: {model_path}")
print(f"Baseline report saved: {report_path}")


Data loaded: train=23,973, dev=2,662
Baseline Macro F1 Score: 0.6129
              precision    recall  f1-score   support

       CLEAN       0.96      0.84      0.89      2180
   OFFENSIVE       0.36      0.52      0.43       212
        HATE       0.42      0.69      0.52       270

    accuracy                           0.80      2662
   macro avg       0.58      0.68      0.61      2662
weighted avg       0.86      0.80      0.82      2662

Baseline artifacts saved: D:\bt_lap_trinh\NLP\Ứng dụng\Project\hate-speech-detection\models\baseline\tfidf_lr.pkl
Baseline report saved: D:\bt_lap_trinh\NLP\Ứng dụng\Project\hate-speech-detection\results\baseline_report.json
